# RAG with LangChain

This notebook rebuilds the manual RAG pipeline implemented in the
previous chapter using LangChain abstractions.

The objective is to understand how LangChain represents and simplifies
the main components of a RAG pipeline:

1. Documents
2. Embeddings
3. Vector storage
4. Retrieval
5. Prompt construction
6. LLM generation

In [3]:
!pip install -q \
    "google-auth==2.49.0" \
    langchain \
    langchain-huggingface \
    langchain-google-genai \
    sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!pip install -q jedi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.1 MB/s eta 0:00:00


In [6]:
!pip check

No broken requirements found.


In [7]:
from google.colab import userdata

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

In [8]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "gemini-3.5-flash-lite"

In [9]:
texts = [
    """
    Amazon S3 is an object storage service designed to store and retrieve
    large amounts of data. Data in S3 is stored as objects inside buckets.
    """,
    """
    Amazon EC2 provides resizable virtual computing capacity in the cloud.
    Users can launch virtual machines called instances.
    """,
    """
    AWS Lambda is a serverless compute service that runs code in response
    to events. It does not require users to provision or manage servers.
    """,
    """
    Amazon RDS is a managed relational database service. It supports
    relational database engines such as PostgreSQL, MySQL and MariaDB.
    """,
    """
    Amazon CloudFront is a content delivery network. It distributes
    content through edge locations to reduce latency.
    """
]

In [10]:
documents = [
    Document(page_content=text.strip())
    for text in texts
]

In [11]:
print(documents[0])

page_content='Amazon S3 is an object storage service designed to store and retrieve
    large amounts of data. Data in S3 is stored as objects inside buckets.'


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={
        "normalize_embeddings": True
    }
)

In [13]:
test_embedding = embeddings.embed_query(
    "Which AWS service stores files?"
)

print(type(test_embedding))
print(len(test_embedding))

<class 'list'>
384


In [14]:
vector_store = InMemoryVectorStore(embeddings)

vector_store.add_documents(documents)

['a7e93ae6-264f-4781-9051-daaf1b7515a7',
 'a5d95d0b-191d-4f47-a8ce-b0f8cc7da556',
 'd0de1195-f7b8-4b49-b3b6-de6721978494',
 'ee71c63b-078c-4e33-8b64-b9db32f4503f',
 'e3337957-4787-4c9a-850c-a5ec8d632b22']

In [15]:
question = "Which AWS service should I use to store files?"

results = vector_store.similarity_search(
    question,
    k=2
)

for doc in results:
    print(doc.page_content)
    print()

Amazon S3 is an object storage service designed to store and retrieve
    large amounts of data. Data in S3 is stored as objects inside buckets.

Amazon RDS is a managed relational database service. It supports
    relational database engines such as PostgreSQL, MySQL and MariaDB.



In [16]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

In the manual version, retrieval was:
  Encode question
    -> dot product
    -> sort scores
    -> top-k indices
    -> get chunks

In [17]:
retriever.invoke(question)

[Document(id='a7e93ae6-264f-4781-9051-daaf1b7515a7', metadata={}, page_content='Amazon S3 is an object storage service designed to store and retrieve\n    large amounts of data. Data in S3 is stored as objects inside buckets.'),
 Document(id='ee71c63b-078c-4e33-8b64-b9db32f4503f', metadata={}, page_content='Amazon RDS is a managed relational database service. It supports\n    relational database engines such as PostgreSQL, MySQL and MariaDB.')]

In [18]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [19]:
retrieved_docs = retriever.invoke(question)

context = format_docs(retrieved_docs)

print(context)

Amazon S3 is an object storage service designed to store and retrieve
    large amounts of data. Data in S3 is stored as objects inside buckets.

Amazon RDS is a managed relational database service. It supports
    relational database engines such as PostgreSQL, MySQL and MariaDB.


In [20]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Answer the user's question using only the provided context.

        If the answer cannot be found in the context, say:
        "The information is not available in the provided context."

        Context:
        {context}
        """
    ),
    (
        "human",
        "{question}"
    )
])

In [21]:
test_prompt = prompt.invoke({
    "context": context,
    "question": question
})

print(test_prompt)

messages=[SystemMessage(content='\n        Answer the user\'s question using only the provided context.\n\n        If the answer cannot be found in the context, say:\n        "The information is not available in the provided context."\n\n        Context:\n        Amazon S3 is an object storage service designed to store and retrieve\n    large amounts of data. Data in S3 is stored as objects inside buckets.\n\nAmazon RDS is a managed relational database service. It supports\n    relational database engines such as PostgreSQL, MySQL and MariaDB.\n        ', additional_kwargs={}, response_metadata={}), HumanMessage(content='Which AWS service should I use to store files?', additional_kwargs={}, response_metadata={})]


In [22]:
api_key = userdata.get("GEMINI_API_KEY")

In [25]:
llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    api_key=api_key,
#    temperature=0
)

In [26]:
response = llm.invoke(
    "Answer in one sentence: what is an API?"
)

print(response.text)

An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and exchange data with each other.


Still explicity orchestrating every stage

In [27]:
retrieved_docs = retriever.invoke(question)

context = format_docs(retrieved_docs)

formatted_prompt = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(formatted_prompt)

print(response.text)

Based on the provided context, you should use Amazon S3 to store files (as it is an object storage service designed to store and retrieve large amounts of data, with data stored as objects inside buckets).


## Actual chain

In [29]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [30]:
answer = rag_chain.invoke(
    "Which AWS service should I use to store files?"
)

print(answer)

Based on the provided context, you should use Amazon S3, which is an object storage service designed to store and retrieve large amounts of data.


In [31]:
answer_no_context = rag_chain.invoke(
    "Which AWS service should I use to run Kubernetes clusters?"
)

print(answer_no_context)

The information is not available in the provided context.
